# Building a first Classification NN

Use the Ames Mutagenicity dataset (from assignment 1A) and build a binary classifier NN. Play with the model parameters. 

For comparison of the NN model performance, consider the performance of other (baseline) classifier models (assignment 1A):
- KNN: Test-Accuracy 0.79, Test-ROC-AUC 0.86
- Decision Tree: Test-Accuracy 0.78, Test-ROC-AUC 0.77
- Random Forest: Test-Accuracy 0.83, Test-ROC-AUC 0.90
- Gradient Boosting: Test-Accuracy 0.77, Test-ROC-AUC 0.85


#### Tasks:
1) Load the dataset `ames_data.csv`. The dataset does not contain any duplicates or NaNs
2) Feature engineering: Calculate various fingerprints from the SMILES strings via mol objects using RDKit(snippet provided for Morgan FPs and MACCS keys)
3) Create feature matrix and target vector. Choose first the MorganFP (Later repeat the process for other fingerprint types). Convert the training and test sets into pytorch tensors.
4) Build your NN (see below for more info)
5) Train your model on the Morgan Fingerprints (and repeat later for other FP types)
6) Evaluate your model's performance and compare to other classifier models
7) Save the model / current state.
8) Respond to the discussion points


#### Note:
The aim of this exercise is to gain a bit of practice in building a simple NN and to see how different parameters and feature engineering influence the model. Maximum accuracy is not the target. 

There is no need to venture too far into the details or more advanced approaches just yet (e.g. batched training would be complete overkill for this assignment - we will discuss that in the next sessions)

0) Import dependencies and datasets

In [23]:
# complete imports if needed for your solution
from pathlib import Path

import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import MACCSkeys
from rdkit.Chem.rdmolops import LayeredFingerprint

from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim

1) Load and investigate the data

In [2]:
df = pd.read_csv("ames_data.csv")
df.head()

,drug_id,smiles,mutagenicity
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7278 entries, 0 to 7277
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   drug_id       7278 non-null   object
 1   smiles        7278 non-null   object
 2   mutagenicity  7278 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 170.7+ KB


In [6]:
df.drop_duplicates() # not needed
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7278 entries, 0 to 7277
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   drug_id       7278 non-null   object
 1   smiles        7278 non-null   object
 2   mutagenicity  7278 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 170.7+ KB


2) Generate different fingerprints (try at least one additional FP type as provided in RDKit and use two different fpSizes on one of them) - all of them will be saved in new columns in the Dataframe.

In [13]:
def smiles_to_mol(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return mol

def morganfp(mol):
    fp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048).GetFingerprint(mol)
    return np.array(fp)

def maccskeys(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)
    return np.array(fp)



def layeredfp(mol):
    fp = LayeredFingerprint(mol)
    return np.array(fp)

fpgens = {
    "MorganFP": morganfp,
    "MACCSkeys": maccskeys,
    "TorsionsFP": layeredfp
}

df["mol"] = df["smiles"].apply(smiles_to_mol)

for name, fpgen in fpgens.items():
    df[name] = df["mol"].apply(fpgen)
df.head()

,drug_id,smiles,mutagenicity,mol,MorganFP,MACCSkeys,TorsionsFP
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1,<rdkit.Chem.rdchem.Mol object at 0x7fef7843dba0>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1,<rdkit.Chem.rdchem.Mol object at 0x7fef7843da50>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ..."
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0,<rdkit.Chem.rdchem.Mol object at 0x7fef7843db30>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ..."
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1,<rdkit.Chem.rdchem.Mol object at 0x7fef783c5040>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1,<rdkit.Chem.rdchem.Mol object at 0x7fef783c50b0>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


3. Create feature matrix and target vector. The snippet below converts the data into numpy arrays. Start with the Morgan Fingerprints (and later return here to apply your modell to different fingerprint types - not all of the fingerprints may have the same length, so you may have to adapt the width of your layers).

Do a train startified test split and convert into pytorch tensors.

In [91]:
def prepare_fp_tensors(df, fp_column, target_column="mutagenicity", test_size=0.2, random_state=42):
    X = np.stack(df[fp_column].values)
    y = df[target_column].to_numpy()

    X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y,
    )

    X_train = torch.tensor(X_train_np, dtype=torch.float32)
    X_test = torch.tensor(X_test_np, dtype=torch.float32)
    y_train = torch.tensor(y_train_np, dtype=torch.float32)
    y_test = torch.tensor(y_test_np, dtype=torch.float32)

    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = prepare_fp_tensors(
    df,
    fp_column="TorsionsFP",
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

(torch.Size([5822, 2048]),
 torch.Size([1456, 2048]),
 torch.Size([5822]),
 torch.Size([1456]))

4) Build the NN - adhere to some robust standard values. Start simple and train the model on Morgan FP first.

Optimise the model parameters based on observed over-/underfitting. Experiment with different width and depth, as well as other model parameters. Explore some options to prevent overfitting, e.g. Early stopping (e.g. manually by limiting the epochs) or dropouts. 

Note: Since the input layer needs a lot of neurons (e.g. 2048 bit in the MFPs), consider shrinking the widht from layer to layer. 

Hint: If you use `BCELoss()` as loss function, combine it with a `sigmoid` activation in the last layer. If you use `BCEWithLogitsLoss()`, do not specify any activation in the forward pass (`x = self.outputlayer(x)`).

In [92]:
class BinClassifierNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super(BinClassifierNN, self).__init__()
        hidden_dim_2 = hidden_dim // 4

        self.hidden_1 = nn.Linear(input_dim, hidden_dim)
        self.hidden_2 = nn.Linear(hidden_dim, hidden_dim_2)
        self.output = nn.Linear(hidden_dim_2, 1)

    def forward(self, x):
        x = torch.relu(self.hidden_1(x))
        x = torch.relu(self.hidden_2(x))
        x = self.output(x)
        return x


In [109]:
# Parameters (change and add as needed)
learning_rate = 0.01
hidden_dim = 256
num_epochs = 100

In [110]:
model = BinClassifierNN(input_dim=X_train.shape[1], hidden_dim=hidden_dim)

# choose a loss function for the classification
criterion = nn.BCEWithLogitsLoss()

# choose an optimizer
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

5. Train the NN. Note that you may have to squeeze the output (`outputs=models(X_train).squeeze`). This will reduce the actual output of the shape ``[N, 1]`` to ``[N]``, which is comparable to y (The final layer naturally produces a column tensor, which is not directly comparable to the 1D target tensor).

In [111]:
for epoch in range(num_epochs):
    
    model.train()
    outputs = model(X_train).squeeze()
    loss = criterion(outputs, y_train)

    optimizer.zero_grad() # clear existing gradients
    loss.backward() # backpropagation of loss function to optimise gradient
    optimizer.step() # update parameters using Adam

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/100], Loss: {loss.item():.4f}')

Epoch [10/100], Loss: 0.6807
Epoch [20/100], Loss: 0.5486
Epoch [30/100], Loss: 0.4372
Epoch [40/100], Loss: 0.3552
Epoch [50/100], Loss: 0.3051
Epoch [60/100], Loss: 0.2522
Epoch [70/100], Loss: 0.1973
Epoch [80/100], Loss: 0.1804
Epoch [90/100], Loss: 0.1521
Epoch [100/100], Loss: 0.1253


6) Evaluate the model. As a first metric, you can use the same loss function to evaluate the model on the test set. For better comparison with methods tested in the assignment 1A (results vide supra), use metrics from scikit-learn (e.g. the accuracy or ROC-AUC score).

Hint: for your prediction you may have to use `squeeze` again to match the target vector in the test set (e.g. ``y_pred = model(X_test).squeeze()``)

In [112]:
model.eval()

with torch.no_grad():
    test_logits = model(X_test).squeeze()
    test_loss = criterion(test_logits, y_test).item()
    test_probs = torch.sigmoid(test_logits)
    y_pred = (test_probs >= 0.5).float()

y_test_np = y_test.numpy()
test_probs_np = test_probs.numpy()
y_pred_np = y_pred.numpy()

test_accuracy = accuracy_score(y_test_np, y_pred_np)
test_roc_auc = roc_auc_score(y_test_np, test_probs_np)
test_confusion = confusion_matrix(y_test_np, y_pred_np)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test ROC-AUC: {test_roc_auc:.4f}")
print("Confusion matrix:")
print(test_confusion)

comparison_df = pd.DataFrame(
    {
        "Model": ["Neural Network", "KNN", "Decision Tree", "Random Forest", "Gradient Boosting"],
        "Accuracy": [test_accuracy, 0.79, 0.78, 0.83, 0.77],
        "ROC-AUC": [test_roc_auc, 0.86, 0.77, 0.90, 0.85],
    }
)

comparison_df

Test loss: 0.7074
Test accuracy: 0.8201
Test ROC-AUC: 0.8822
Confusion matrix:
[[476 185]
 [ 77 718]]


,Model,Accuracy,ROC-AUC
0,Neural Network,0.820055,0.882222
1,KNN,0.790000,0.860000
2,Decision Tree,0.780000,0.770000
3,Random Forest,0.830000,0.900000
4,Gradient Boosting,0.770000,0.850000


7) Research how you can save the model / current state for later reuse. What are different options here? How can it be loaded again?

In [52]:
if "Model_results_df" not in globals():
    Model_results_df = pd.DataFrame(columns=[
        "model_state_dict",
        "input_dim",
        "hidden_dim",
        "learning_rate",
        "num_epochs",
        "Test loss",
        "Test accuracy",
        "Test ROC-AUC"
    ])

Model_results_df

,model_state_dict,input_dim,hidden_dim,learning_rate,num_epochs,Test loss,Test accuracy,Test ROC-AUC


In [113]:
# Recommended: save the model parameters (state_dict) and useful metadata
save_dir = Path("saved_models")
save_dir.mkdir(exist_ok=True)

checkpoint_path = save_dir / "TorsionsFP_nn_state_dict_4.pt"

saved_hyperparameters = {
    "model_state_dict": checkpoint_path,
    "input_dim": X_train.shape[1],
    "hidden_dim": hidden_dim,
    "learning_rate": learning_rate,
    "num_epochs": num_epochs,
    "Test loss": test_loss,
    "Test accuracy": test_accuracy,
    "Test ROC-AUC": test_roc_auc,
}

checkpoint_metadata = {
    key: value for key, value in saved_hyperparameters.items() if key != "model_state_dict"
}

torch.save(
    {
        "model_state_dict": model.state_dict(),
        **checkpoint_metadata,
    },
    checkpoint_path,
)

Model_results_df = pd.concat(
    [Model_results_df, pd.DataFrame([saved_hyperparameters])],
    ignore_index=True,
)

print(f"Model state saved to: {checkpoint_path}")

# Load again later
checkpoint = torch.load(checkpoint_path)
loaded_model = BinClassifierNN(
    input_dim=checkpoint["input_dim"],
    hidden_dim=checkpoint["hidden_dim"],
)
loaded_model.load_state_dict(checkpoint["model_state_dict"])
loaded_model.eval()

Model_results_df


Model state saved to: saved_models/TorsionsFP_nn_state_dict_4.pt


,model_state_dict,input_dim,hidden_dim,learning_rate,num_epochs,Test loss,Test accuracy,Test ROC-AUC
0,saved_models/morgan_nn_state_dict_2.pt,2048,512,0.001,100,1.578948,0.782967,0.844867
1,saved_models/morgan_nn_state_dict_3.pt,2048,512,0.010,100,3.175755,0.784341,0.844717
2,saved_models/morgan_nn_state_dict_3.pt,2048,256,0.001,100,1.227486,0.771978,0.836062
3,saved_models/morgan_nn_state_dict_3.pt,2048,256,0.001,500,2.223878,0.776786,0.829974
4,saved_models/MACCSkeys_nn_state_dict_1.pt,167,256,0.001,500,0.863096,0.817995,0.884900
5,saved_models/MACCSkeys_nn_state_dict_1.pt,167,128,0.001,100,0.445036,0.802885,0.875194
6,saved_models/MACCSkeys_nn_state_dict_3.pt,167,128,0.010,100,0.701615,0.812500,0.874461
7,saved_models/MACCSkeys_nn_state_dict_4.pt,167,128,0.010,500,1.508415,0.811126,0.870105
8,saved_models/TorsionsFP_nn_state_dict_1.pt,2048,128,0.001,100,0.489136,0.826923,0.889751
9,saved_models/TorsionsFP_nn_state_dict_2.pt,2048,512,0.001,100,0.653813,0.837225,0.892195


#### 8) Discussion points
1) How did your model compare to other simple ML classifiers (all used the Morgan FPs)? Discuss!
2) Did you observe any difference between different fingerprint types?
3) Did the fingerprint size impact the model prediction? What message is to be learned from this?
4) What were some model parameters for decent performance depending on the fingerprint type? 
5) Was overfitting a problem? What approaches did you apply to limit that issue? What else would be possible
6) Consider the target "mutagenicity" in the context of molecular structure. What does noise mean here? How could you use such a predictive model in the lab? What other data-driven tools could be interesting in this QSAR context?
7) Why is exporting a full model usually not recommended?

#### Short Answers
1. The NN was decent, but usually below Random Forest and slightly below KNN on MorganFP.
2. Yes. TorsionsFP performed best, MACCSkeys was also strong, MorganFP was weakest in these runs.
3. FP size was not tested explicitly here, so no clear conclusion from this notebook alone.
4. Good settings were mostly `100` epochs, `lr=0.001`, and hidden sizes around `256-512`, depending on FP type.
5. Overfitting was possible, especially in longer runs. Simpler models and limiting epochs helped.
6. Noise means imperfect experimental labels or assay variability. The model could help pre-screen compounds in the lab.
7. Exporting the full model is less robust; saving the `state_dict` is safer and more portable.
